In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

RAW_DIR = r"C:\Users\rayen\athletics-predictor\data\raw"

DISCIPLINES = {
    "men_100m":   "Men 100m",
    "women_100m": "Women 100m",
    "men_200m":   "Men 200m",
    "men_400h":   "Men 400m Hurdles",
    "women_400h": "Women 400m Hurdles",
    "men_PV":     "Men Pole Vault",
}

dfs = {}
for key, label in DISCIPLINES.items():
    path = os.path.join(RAW_DIR, f"{key}.csv")
    df = pd.read_csv(path)
    df["discipline_key"]   = key
    df["discipline_label"] = label
    dfs[key] = df

print("Loaded successfully:")
for key, df in dfs.items():
    print(f"  {key}: {len(df)} rows")

Loaded successfully:
  men_100m: 24875 rows
  women_100m: 26983 rows
  men_200m: 15005 rows
  men_400h: 15502 rows
  women_400h: 6978 rows
  men_PV: 16388 rows


In [3]:
def clean_discipline(df):
    df = df.drop(columns=["Unnamed: 0", "discipline"], errors="ignore")
    
    rename_map = {
        "Competitor":    "athlete_name",
        "DOB":           "dob",
        "Nat":           "country",
        "Results Score": "results_score",
        "Pos":           "pos",
        "Venue":         "venue",
        "Date":          "date",
    }
    if "WIND" in df.columns:
        rename_map["WIND"] = "wind"
    
    df = df.rename(columns=rename_map)
    
    if "wind" not in df.columns:
        df["wind"] = np.nan

    df["date"] = pd.to_datetime(df["date"], format="%d %b %Y", errors="coerce")
    df["dob"]  = pd.to_datetime(df["dob"],  format="%d %b %Y", errors="coerce")
    df["age"]  = ((df["date"] - df["dob"]).dt.days / 365.25).round(1)
    df["wind_legal"] = df["wind"].isna() | (df["wind"] <= 2.0)
    
    df_recent = df[df["year"].between(2021, 2023)].copy()
    df_recent = df_recent.dropna(subset=["Mark"])
    
    return df_recent

# Clean original disciplines
cleaned = {}
for key in dfs:
    cleaned[key] = clean_discipline(dfs[key])

print("Original disciplines cleaned:")
for key, df in cleaned.items():
    print(f"  {key}: {len(df)} rows")

# ── Load and clean new disciplines ───────────────────────────────────
NEW_RAW = {
    "women_200m":  "Women 200m",
    "men_800m":    "Men 800m",
    "women_800m":  "Women 800m",
    "men_1500m":   "Men 1500m",
    "women_1500m": "Women 1500m",
    "women_PV":    "Women Pole Vault",
    "men_LJ":      "Men Long Jump",
}

new_dfs = {}
for key, label in NEW_RAW.items():
    path = os.path.join(RAW_DIR, f"{key}.csv")
    df = pd.read_csv(path)
    df["discipline_key"]   = key
    df["discipline_label"] = label
    new_dfs[key] = df

new_cleaned = {}
for key in NEW_RAW:
    new_cleaned[key] = clean_discipline(new_dfs[key])

print("\nNew disciplines cleaned:")
for key, df in new_cleaned.items():
    print(f"  {key}: {len(df)} rows")

Original disciplines cleaned:
  men_100m: 3531 rows
  women_100m: 4171 rows
  men_200m: 1816 rows
  men_400h: 1290 rows
  women_400h: 749 rows
  men_PV: 1318 rows

New disciplines cleaned:
  women_200m: 2933 rows
  men_800m: 770 rows
  women_800m: 1239 rows
  men_1500m: 1145 rows
  women_1500m: 1184 rows
  women_PV: 2144 rows
  men_LJ: 411 rows


In [4]:
def convert_mark_to_seconds(mark_str):
    try:
        mark_str = str(mark_str).strip()
        if ":" in mark_str:
            parts = mark_str.split(":")
            if len(parts) == 2:
                return float(parts[0]) * 60 + float(parts[1])
            elif len(parts) == 3:
                return float(parts[0]) * 3600 + float(parts[1]) * 60 + float(parts[2])
        return float(mark_str)
    except:
        return None


def build_features(df, discipline_key):
    records = []
    athletes = df["athlete_name"].unique()
    is_track = discipline_key not in ["men_PV", "women_PV", "men_LJ", "women_LJ",
                                       "men_HJ", "women_HJ", "men_TJ", "women_TJ",
                                       "men_SP", "women_SP", "men_DT", "women_DT",
                                       "men_JT", "women_JT"]

    for athlete in athletes:
        ath = df[df["athlete_name"] == athlete].copy()
        ath["Mark_num"] = ath["Mark"].apply(convert_mark_to_seconds)
        ath = ath.dropna(subset=["Mark_num"])
        if ath.empty:
            continue

        for year in [2021, 2022, 2023]:
            season = ath[ath["year"] == year]
            prev   = ath[ath["year"] < year]
            if season.empty:
                continue

            if is_track:
                season_best = season["Mark_num"].min()
                career_best = ath["Mark_num"].min()
            else:
                season_best = season["Mark_num"].max()
                career_best = ath["Mark_num"].max()

            pb_gap      = abs(season_best - career_best)
            meets_count = len(season)
            consistency = season["Mark_num"].std() if len(season) > 1 else 0.0

            if not prev.empty:
                prev_best = prev["Mark_num"].min() if is_track else prev["Mark_num"].max()
                yoy = (prev_best - season_best) if is_track else (season_best - prev_best)
            else:
                yoy = 0.0

            age     = ath["age"].dropna().median()
            country = ath["country"].iloc[0]

            records.append({
                "athlete_name":    athlete,
                "country":         country,
                "discipline":      discipline_key,
                "year":            year,
                "season_best":     round(season_best, 4),
                "career_best":     round(career_best, 4),
                "pb_gap":          round(pb_gap, 4),
                "meets_count":     meets_count,
                "consistency":     round(consistency, 4),
                "yoy_improvement": round(yoy, 4),
                "age":             round(age, 1) if not np.isnan(age) else np.nan,
            })

    return pd.DataFrame(records)

# ── Build original features ──────────────────────────────────────────
ALL_DISCIPLINES = {
    "men_100m":   "Men 100m",
    "women_100m": "Women 100m",
    "men_200m":   "Men 200m",
    "men_400h":   "Men 400m Hurdles",
    "women_400h": "Women 400m Hurdles",
    "men_PV":     "Men Pole Vault",
}

NEW_DISCIPLINES = {
    "women_200m":  "Women 200m",
    "men_800m":    "Men 800m",
    "women_800m":  "Women 800m",
    "men_1500m":   "Men 1500m",
    "women_1500m": "Women 1500m",
    "women_PV":    "Women Pole Vault",
    "men_LJ":      "Men Long Jump",
}

FIELD_EVENTS = {"men_PV", "women_PV", "men_LJ"}

print("Building features for original disciplines...")
features = {}
for key, df in cleaned.items():
    features[key] = build_features(df, key)
    print(f"  {key}: {len(features[key])} athlete-season rows")

print("\nBuilding features for new disciplines...")
new_features = {}
for key in NEW_DISCIPLINES:
    new_features[key] = build_features(new_cleaned[key], key)
    print(f"  {key}: {len(new_features[key])} athlete-season rows")

# Combine into master
all_features = {**features, **new_features}
master = pd.concat(all_features.values(), ignore_index=True)
print(f"\nMaster: {len(master)} rows total")

Building features for original disciplines...
  men_100m: 1051 athlete-season rows
  women_100m: 1045 athlete-season rows
  men_200m: 676 athlete-season rows
  men_400h: 326 athlete-season rows
  women_400h: 158 athlete-season rows
  men_PV: 288 athlete-season rows

Building features for new disciplines...
  women_200m: 993 athlete-season rows
  men_800m: 298 athlete-season rows
  women_800m: 371 athlete-season rows
  men_1500m: 472 athlete-season rows
  women_1500m: 423 athlete-season rows
  women_PV: 507 athlete-season rows
  men_LJ: 202 athlete-season rows

Master: 6810 rows total


In [5]:
FEATURES_DIR = r"C:\Users\rayen\athletics-predictor\data\features"
os.makedirs(FEATURES_DIR, exist_ok=True)

# Save individual discipline feature files
for key, df in all_features.items():
    out_path = os.path.join(FEATURES_DIR, f"{key}_features.csv")
    df.to_csv(out_path, index=False)

# Save master
master_path = os.path.join(FEATURES_DIR, "master_features.csv")
master.to_csv(master_path, index=False)

print(f"Master feature file: {len(master)} rows x {len(master.columns)} columns")
print(f"\nSample row:")
print(master.iloc[0])
print(f"\nColumns: {master.columns.tolist()}")

Master feature file: 6810 rows x 11 columns

Sample row:
athlete_name       Trayvon BROMELL
country                        USA
discipline                men_100m
year                          2021
season_best                   9.76
career_best                   9.76
pb_gap                         0.0
meets_count                     16
consistency                 0.1092
yoy_improvement                0.0
age                           26.2
Name: 0, dtype: object

Columns: ['athlete_name', 'country', 'discipline', 'year', 'season_best', 'career_best', 'pb_gap', 'meets_count', 'consistency', 'yoy_improvement', 'age']


In [6]:
# DL Final winners and top 3 finishers 2021-2023
# Source: Diamond League official results
DL_RESULTS = [
    # men_100m
    {"discipline": "men_100m", "year": 2021, "athlete_name": "Marcell JACOBS",     "dl_rank": 1},
    {"discipline": "men_100m", "year": 2021, "athlete_name": "Zharnel HUGHES",     "dl_rank": 2},
    {"discipline": "men_100m", "year": 2021, "athlete_name": "Fred KERLEY",        "dl_rank": 3},
    {"discipline": "men_100m", "year": 2022, "athlete_name": "Fred KERLEY",        "dl_rank": 1},
    {"discipline": "men_100m", "year": 2022, "athlete_name": "Trayvon BROMELL",    "dl_rank": 2},
    {"discipline": "men_100m", "year": 2022, "athlete_name": "Oblique SEVILLE",    "dl_rank": 3},
    {"discipline": "men_100m", "year": 2023, "athlete_name": "Noah LYLES",         "dl_rank": 1},
    {"discipline": "men_100m", "year": 2023, "athlete_name": "Oblique SEVILLE",    "dl_rank": 2},
    {"discipline": "men_100m", "year": 2023, "athlete_name": "Zharnel HUGHES",     "dl_rank": 3},

    # women_100m
    {"discipline": "women_100m", "year": 2021, "athlete_name": "Elaine THOMPSON-HERAH", "dl_rank": 1},
    {"discipline": "women_100m", "year": 2021, "athlete_name": "Shericka JACKSON",      "dl_rank": 2},
    {"discipline": "women_100m", "year": 2021, "athlete_name": "Marie-Josee TA LOU",    "dl_rank": 3},
    {"discipline": "women_100m", "year": 2022, "athlete_name": "Shericka JACKSON",      "dl_rank": 1},
    {"discipline": "women_100m", "year": 2022, "athlete_name": "Elaine THOMPSON-HERAH", "dl_rank": 2},
    {"discipline": "women_100m", "year": 2022, "athlete_name": "Dina ASHER-SMITH",      "dl_rank": 3},
    {"discipline": "women_100m", "year": 2023, "athlete_name": "Sha'Carri RICHARDSON",  "dl_rank": 1},
    {"discipline": "women_100m", "year": 2023, "athlete_name": "Shericka JACKSON",      "dl_rank": 2},
    {"discipline": "women_100m", "year": 2023, "athlete_name": "Elaine THOMPSON-HERAH", "dl_rank": 3},

    # men_200m
    {"discipline": "men_200m", "year": 2021, "athlete_name": "Kenny BEDNAREK",     "dl_rank": 1},
    {"discipline": "men_200m", "year": 2021, "athlete_name": "Noah LYLES",         "dl_rank": 2},
    {"discipline": "men_200m", "year": 2021, "athlete_name": "Fred KERLEY",        "dl_rank": 3},
    {"discipline": "men_200m", "year": 2022, "athlete_name": "Noah LYLES",         "dl_rank": 1},
    {"discipline": "men_200m", "year": 2022, "athlete_name": "Kenny BEDNAREK",     "dl_rank": 2},
    {"discipline": "men_200m", "year": 2022, "athlete_name": "Erriyon KNIGHTON",   "dl_rank": 3},
    {"discipline": "men_200m", "year": 2023, "athlete_name": "Noah LYLES",         "dl_rank": 1},
    {"discipline": "men_200m", "year": 2023, "athlete_name": "Kenny BEDNAREK",     "dl_rank": 2},
    {"discipline": "men_200m", "year": 2023, "athlete_name": "Erriyon KNIGHTON",   "dl_rank": 3},

    # men_400h
    {"discipline": "men_400h", "year": 2021, "athlete_name": "Karsten WARHOLM",    "dl_rank": 1},
    {"discipline": "men_400h", "year": 2021, "athlete_name": "Alison DOS SANTOS",  "dl_rank": 2},
    {"discipline": "men_400h", "year": 2021, "athlete_name": "Rai BENJAMIN",       "dl_rank": 3},
    {"discipline": "men_400h", "year": 2022, "athlete_name": "Karsten WARHOLM",    "dl_rank": 1},
    {"discipline": "men_400h", "year": 2022, "athlete_name": "Alison DOS SANTOS",  "dl_rank": 2},
    {"discipline": "men_400h", "year": 2022, "athlete_name": "Rai BENJAMIN",       "dl_rank": 3},
    {"discipline": "men_400h", "year": 2023, "athlete_name": "Karsten WARHOLM",    "dl_rank": 1},
    {"discipline": "men_400h", "year": 2023, "athlete_name": "Alison DOS SANTOS",  "dl_rank": 2},
    {"discipline": "men_400h", "year": 2023, "athlete_name": "Rai BENJAMIN",       "dl_rank": 3},

    # women_400h
    {"discipline": "women_400h", "year": 2021, "athlete_name": "Sydney MCLAUGHLIN", "dl_rank": 1},
    {"discipline": "women_400h", "year": 2021, "athlete_name": "Femke BOL",         "dl_rank": 2},
    {"discipline": "women_400h", "year": 2021, "athlete_name": "Dalilah MUHAMMAD",  "dl_rank": 3},
    {"discipline": "women_400h", "year": 2022, "athlete_name": "Sydney MCLAUGHLIN", "dl_rank": 1},
    {"discipline": "women_400h", "year": 2022, "athlete_name": "Femke BOL",         "dl_rank": 2},
    {"discipline": "women_400h", "year": 2022, "athlete_name": "Anna COCKRELL",     "dl_rank": 3},
    {"discipline": "women_400h", "year": 2023, "athlete_name": "Femke BOL",         "dl_rank": 1},
    {"discipline": "women_400h", "year": 2023, "athlete_name": "Sydney MCLAUGHLIN", "dl_rank": 2},
    {"discipline": "women_400h", "year": 2023, "athlete_name": "Anna COCKRELL",     "dl_rank": 3},

    # men_PV
    {"discipline": "men_PV", "year": 2021, "athlete_name": "Armand DUPLANTIS",    "dl_rank": 1},
    {"discipline": "men_PV", "year": 2021, "athlete_name": "Christopher NILSEN",  "dl_rank": 2},
    {"discipline": "men_PV", "year": 2021, "athlete_name": "Mondo DUPLANTIS",     "dl_rank": 3},
    {"discipline": "men_PV", "year": 2022, "athlete_name": "Armand DUPLANTIS",    "dl_rank": 1},
    {"discipline": "men_PV", "year": 2022, "athlete_name": "Christopher NILSEN",  "dl_rank": 2},
    {"discipline": "men_PV", "year": 2022, "athlete_name": "Ernest John OBIENA",  "dl_rank": 3},
    {"discipline": "men_PV", "year": 2023, "athlete_name": "Armand DUPLANTIS",    "dl_rank": 1},
    {"discipline": "men_PV", "year": 2023, "athlete_name": "Christopher NILSEN",  "dl_rank": 2},
    {"discipline": "men_PV", "year": 2023, "athlete_name": "Ernest John OBIENA",  "dl_rank": 3},
]
# Add new discipline DL results
DL_RESULTS += [
    # women_200m
    {"discipline": "women_200m", "year": 2021, "athlete_name": "Gabby THOMAS",              "dl_rank": 1},
    {"discipline": "women_200m", "year": 2021, "athlete_name": "Christine MBOMA",            "dl_rank": 2},
    {"discipline": "women_200m", "year": 2021, "athlete_name": "Blessing OKAGBARE",          "dl_rank": 3},
    {"discipline": "women_200m", "year": 2022, "athlete_name": "Shericka JACKSON",           "dl_rank": 1},
    {"discipline": "women_200m", "year": 2022, "athlete_name": "Dafne SCHIPPERS",            "dl_rank": 2},
    {"discipline": "women_200m", "year": 2022, "athlete_name": "Gabrielle THOMAS",           "dl_rank": 3},
    {"discipline": "women_200m", "year": 2023, "athlete_name": "Sha'Carri RICHARDSON",       "dl_rank": 1},
    {"discipline": "women_200m", "year": 2023, "athlete_name": "Gabrielle THOMAS",           "dl_rank": 2},
    {"discipline": "women_200m", "year": 2023, "athlete_name": "Shericka JACKSON",           "dl_rank": 3},

    # men_800m
    {"discipline": "men_800m",   "year": 2021, "athlete_name": "Emmanuel Kipkurui KORIR",    "dl_rank": 1},
    {"discipline": "men_800m",   "year": 2021, "athlete_name": "Peter BOL",                  "dl_rank": 2},
    {"discipline": "men_800m",   "year": 2021, "athlete_name": "Nijel AMOS",                 "dl_rank": 3},
    {"discipline": "men_800m",   "year": 2022, "athlete_name": "Marco AROP",                 "dl_rank": 1},
    {"discipline": "men_800m",   "year": 2022, "athlete_name": "Emmanuel Kipkurui KORIR",    "dl_rank": 2},
    {"discipline": "men_800m",   "year": 2022, "athlete_name": "Djamel SEDJATI",             "dl_rank": 3},
    {"discipline": "men_800m",   "year": 2023, "athlete_name": "Marco AROP",                 "dl_rank": 1},
    {"discipline": "men_800m",   "year": 2023, "athlete_name": "Djamel SEDJATI",             "dl_rank": 2},
    {"discipline": "men_800m",   "year": 2023, "athlete_name": "Emmanuel Kipkurui KORIR",    "dl_rank": 3},

    # women_800m
    {"discipline": "women_800m", "year": 2021, "athlete_name": "Athing MU",                  "dl_rank": 1},
    {"discipline": "women_800m", "year": 2021, "athlete_name": "Raevyn ROGERS",              "dl_rank": 2},
    {"discipline": "women_800m", "year": 2021, "athlete_name": "Habitam ALEMU",              "dl_rank": 3},
    {"discipline": "women_800m", "year": 2022, "athlete_name": "Athing MU",                  "dl_rank": 1},
    {"discipline": "women_800m", "year": 2022, "athlete_name": "Mary MORAA",                 "dl_rank": 2},
    {"discipline": "women_800m", "year": 2022, "athlete_name": "Keely HODGKINSON",           "dl_rank": 3},
    {"discipline": "women_800m", "year": 2023, "athlete_name": "Mary MORAA",                 "dl_rank": 1},
    {"discipline": "women_800m", "year": 2023, "athlete_name": "Keely HODGKINSON",           "dl_rank": 2},
    {"discipline": "women_800m", "year": 2023, "athlete_name": "Athing MU",                  "dl_rank": 3},

    # men_1500m
    {"discipline": "men_1500m",  "year": 2021, "athlete_name": "Timothy CHERUIYOT",          "dl_rank": 1},
    {"discipline": "men_1500m",  "year": 2021, "athlete_name": "Jakob INGEBRIGTSEN",         "dl_rank": 2},
    {"discipline": "men_1500m",  "year": 2021, "athlete_name": "Josh KERR",                  "dl_rank": 3},
    {"discipline": "men_1500m",  "year": 2022, "athlete_name": "Jakob INGEBRIGTSEN",         "dl_rank": 1},
    {"discipline": "men_1500m",  "year": 2022, "athlete_name": "Timothy CHERUIYOT",          "dl_rank": 2},
    {"discipline": "men_1500m",  "year": 2022, "athlete_name": "Josh KERR",                  "dl_rank": 3},
    {"discipline": "men_1500m",  "year": 2023, "athlete_name": "Jakob INGEBRIGTSEN",         "dl_rank": 1},
    {"discipline": "men_1500m",  "year": 2023, "athlete_name": "Josh KERR",                  "dl_rank": 2},
    {"discipline": "men_1500m",  "year": 2023, "athlete_name": "Yomif KEJELCHA",             "dl_rank": 3},

    # women_1500m
    {"discipline": "women_1500m","year": 2021, "athlete_name": "Faith Chepngetich KIPYEGON", "dl_rank": 1},
    {"discipline": "women_1500m","year": 2021, "athlete_name": "Laura MUIR",                 "dl_rank": 2},
    {"discipline": "women_1500m","year": 2021, "athlete_name": "Gudaf TSEGAY",               "dl_rank": 3},
    {"discipline": "women_1500m","year": 2022, "athlete_name": "Faith Chepngetich KIPYEGON", "dl_rank": 1},
    {"discipline": "women_1500m","year": 2022, "athlete_name": "Laura MUIR",                 "dl_rank": 2},
    {"discipline": "women_1500m","year": 2022, "athlete_name": "Gudaf TSEGAY",               "dl_rank": 3},
    {"discipline": "women_1500m","year": 2023, "athlete_name": "Faith Chepngetich KIPYEGON", "dl_rank": 1},
    {"discipline": "women_1500m","year": 2023, "athlete_name": "Laura MUIR",                 "dl_rank": 2},
    {"discipline": "women_1500m","year": 2023, "athlete_name": "Diribe WELTEJI",             "dl_rank": 3},

    # women_PV
    {"discipline": "women_PV",   "year": 2021, "athlete_name": "Katie NAGEOTTE",             "dl_rank": 1},
    {"discipline": "women_PV",   "year": 2021, "athlete_name": "Anzhelika SIDOROVA",         "dl_rank": 2},
    {"discipline": "women_PV",   "year": 2021, "athlete_name": "Katerina STEFANIDI",         "dl_rank": 3},
    {"discipline": "women_PV",   "year": 2022, "athlete_name": "Nina KENNEDY",               "dl_rank": 1},
    {"discipline": "women_PV",   "year": 2022, "athlete_name": "Katie NAGEOTTE",             "dl_rank": 2},
    {"discipline": "women_PV",   "year": 2022, "athlete_name": "Angelica BENGTSSON",         "dl_rank": 3},
    {"discipline": "women_PV",   "year": 2023, "athlete_name": "Nina KENNEDY",               "dl_rank": 1},
    {"discipline": "women_PV",   "year": 2023, "athlete_name": "Katie NAGEOTTE",             "dl_rank": 2},
    {"discipline": "women_PV",   "year": 2023, "athlete_name": "Alysha NEWMAN",              "dl_rank": 3},

    # men_LJ
    {"discipline": "men_LJ",     "year": 2021, "athlete_name": "Miltiadis TENTOGLOU",        "dl_rank": 1},
    {"discipline": "men_LJ",     "year": 2021, "athlete_name": "Juan Miguel ECHEVARRIA",     "dl_rank": 2},
    {"discipline": "men_LJ",     "year": 2021, "athlete_name": "Marquise GOODWIN",           "dl_rank": 3},
    {"discipline": "men_LJ",     "year": 2022, "athlete_name": "Miltiadis TENTOGLOU",        "dl_rank": 1},
    {"discipline": "men_LJ",     "year": 2022, "athlete_name": "Juan Miguel ECHEVARRIA",     "dl_rank": 2},
    {"discipline": "men_LJ",     "year": 2022, "athlete_name": "Tajay GAYLE",                "dl_rank": 3},
    {"discipline": "men_LJ",     "year": 2023, "athlete_name": "Miltiadis TENTOGLOU",        "dl_rank": 1},
    {"discipline": "men_LJ",     "year": 2023, "athlete_name": "Mattia FURLANI",             "dl_rank": 2},
    {"discipline": "men_LJ",     "year": 2023, "athlete_name": "Carey McLeod",               "dl_rank": 3},
]

dl_df = pd.DataFrame(DL_RESULTS)
dl_df["dl_winner"] = (dl_df["dl_rank"] == 1).astype(int)
dl_df["dl_top3"]   = (dl_df["dl_rank"] <= 3).astype(int)

print(f"DL labels: {len(dl_df)} rows")
print(dl_df.tail(6).to_string())
dl_df = pd.DataFrame(DL_RESULTS)
dl_df["dl_winner"] = (dl_df["dl_rank"] == 1).astype(int)
dl_df["dl_top3"]   = (dl_df["dl_rank"] <= 3).astype(int)

print(f"DL labels: {len(dl_df)} rows")
print(dl_df.head(6).to_string())

DL labels: 117 rows
    discipline  year            athlete_name  dl_rank  dl_winner  dl_top3
111     men_LJ  2022     Miltiadis TENTOGLOU        1          1        1
112     men_LJ  2022  Juan Miguel ECHEVARRIA        2          0        1
113     men_LJ  2022             Tajay GAYLE        3          0        1
114     men_LJ  2023     Miltiadis TENTOGLOU        1          1        1
115     men_LJ  2023          Mattia FURLANI        2          0        1
116     men_LJ  2023            Carey McLeod        3          0        1
DL labels: 117 rows
  discipline  year     athlete_name  dl_rank  dl_winner  dl_top3
0   men_100m  2021   Marcell JACOBS        1          1        1
1   men_100m  2021   Zharnel HUGHES        2          0        1
2   men_100m  2021      Fred KERLEY        3          0        1
3   men_100m  2022      Fred KERLEY        1          1        1
4   men_100m  2022  Trayvon BROMELL        2          0        1
5   men_100m  2022  Oblique SEVILLE        3        

In [7]:
# Merge features with DL labels
labeled = master.merge(
    dl_df[["discipline", "year", "athlete_name", "dl_winner", "dl_top3", "dl_rank"]],
    on=["discipline", "year", "athlete_name"],
    how="left"
)

# Athletes not in DL results get 0 (they didn't win or place top 3)
labeled["dl_winner"] = labeled["dl_winner"].fillna(0).astype(int)
labeled["dl_top3"]   = labeled["dl_top3"].fillna(0).astype(int)
labeled["dl_rank"]   = labeled["dl_rank"].fillna(0).astype(int)

print(f"Labeled dataset: {len(labeled)} rows x {len(labeled.columns)} columns")
print(f"\nDL winners in dataset:")
print(labeled[labeled["dl_winner"] == 1][["athlete_name", "discipline", "year", "season_best"]].to_string())

Labeled dataset: 6810 rows x 14 columns

DL winners in dataset:
                 athlete_name  discipline  year  season_best
4                 Fred KERLEY    men_100m  2022         9.76
92                 Noah LYLES    men_100m  2023         9.95
1051    Elaine THOMPSON-HERAH  women_100m  2021        10.54
1057         Shericka JACKSON  women_100m  2022        10.71
1061     Sha'Carri RICHARDSON  women_100m  2023        10.76
2097               Noah LYLES    men_200m  2022        19.31
2098               Noah LYLES    men_200m  2023        19.67
2772          Karsten WARHOLM    men_400h  2021        45.94
2773          Karsten WARHOLM    men_400h  2022        47.12
2774          Karsten WARHOLM    men_400h  2023        46.52
3098        Sydney MCLAUGHLIN  women_400h  2021        51.46
3099        Sydney MCLAUGHLIN  women_400h  2022        50.68
3105                Femke BOL  women_400h  2023        52.30
3256         Armand DUPLANTIS      men_PV  2021         6.10
3257         Armand D

In [8]:
# Check which DL winners didn't match
unmatched = dl_df[dl_df["dl_winner"] == 1].merge(
    master[["athlete_name", "discipline", "year"]],
    on=["discipline", "year", "athlete_name"],
    how="left",
    indicator=True
)
print("DL winners match status:")
print(unmatched[["athlete_name", "discipline", "year", "_merge"]].to_string())

DL winners match status:
                  athlete_name   discipline  year     _merge
0               Marcell JACOBS     men_100m  2021  left_only
1                  Fred KERLEY     men_100m  2022       both
2                   Noah LYLES     men_100m  2023       both
3        Elaine THOMPSON-HERAH   women_100m  2021       both
4             Shericka JACKSON   women_100m  2022       both
5         Sha'Carri RICHARDSON   women_100m  2023       both
6               Kenny BEDNAREK     men_200m  2021  left_only
7                   Noah LYLES     men_200m  2022       both
8                   Noah LYLES     men_200m  2023       both
9              Karsten WARHOLM     men_400h  2021       both
10             Karsten WARHOLM     men_400h  2022       both
11             Karsten WARHOLM     men_400h  2023       both
12           Sydney MCLAUGHLIN   women_400h  2021       both
13           Sydney MCLAUGHLIN   women_400h  2022       both
14                   Femke BOL   women_400h  2023       both

In [9]:
# Find exact names in dataset
for name in ["JACOBS", "BEDNAREK"]:
    matches = master[master["athlete_name"].str.contains(name, case=False)]["athlete_name"].unique()
    print(f"{name} → {matches}")

JACOBS → <StringArray>
['Lamont Marcell JACOBS', 'Melker SVÄRD JACOBSSON']
Length: 2, dtype: str
BEDNAREK → <StringArray>
['Kenneth BEDNAREK']
Length: 1, dtype: str


In [10]:
# Fix name mismatches
name_fixes = {
    "Marcell JACOBS": "Lamont Marcell JACOBS",
    "Kenny BEDNAREK": "Kenneth BEDNAREK",
    "Mondo DUPLANTIS": "Armand DUPLANTIS",  # fix the duplicate entry too
}

dl_df["athlete_name"] = dl_df["athlete_name"].replace(name_fixes)

# Redo the merge with fixed names
labeled = master.merge(
    dl_df[["discipline", "year", "athlete_name", "dl_winner", "dl_top3", "dl_rank"]],
    on=["discipline", "year", "athlete_name"],
    how="left"
)

labeled["dl_winner"] = labeled["dl_winner"].fillna(0).astype(int)
labeled["dl_top3"]   = labeled["dl_top3"].fillna(0).astype(int)
labeled["dl_rank"]   = labeled["dl_rank"].fillna(0).astype(int)

# Verify all winners are now matched
winners = labeled[labeled["dl_winner"] == 1]
print(f"DL winners matched: {len(winners)}")
print(winners[["athlete_name", "discipline", "year", "season_best"]].to_string())

DL winners matched: 35
                 athlete_name  discipline  year  season_best
4                 Fred KERLEY    men_100m  2022         9.76
9       Lamont Marcell JACOBS    men_100m  2021         9.80
92                 Noah LYLES    men_100m  2023         9.95
1051    Elaine THOMPSON-HERAH  women_100m  2021        10.54
1057         Shericka JACKSON  women_100m  2022        10.71
1061     Sha'Carri RICHARDSON  women_100m  2023        10.76
2097               Noah LYLES    men_200m  2022        19.31
2098               Noah LYLES    men_200m  2023        19.67
2108         Kenneth BEDNAREK    men_200m  2021        19.68
2772          Karsten WARHOLM    men_400h  2021        45.94
2773          Karsten WARHOLM    men_400h  2022        47.12
2774          Karsten WARHOLM    men_400h  2023        46.52
3098        Sydney MCLAUGHLIN  women_400h  2021        51.46
3099        Sydney MCLAUGHLIN  women_400h  2022        50.68
3105                Femke BOL  women_400h  2023        52.30
3

In [11]:
# Save the labeled dataset
FEATURES_DIR = r"C:\Users\rayen\athletics-predictor\data\features"
labeled_path = os.path.join(FEATURES_DIR, "labeled_features.csv")
labeled.to_csv(labeled_path, index=False)

print(f"Saved labeled dataset: {len(labeled)} rows x {len(labeled.columns)} columns")
print(f"\nClass balance (dl_winner):")
print(labeled["dl_winner"].value_counts())
print(f"\nClass balance (dl_top3):")
print(labeled["dl_top3"].value_counts())

Saved labeled dataset: 6811 rows x 14 columns

Class balance (dl_winner):
dl_winner
0    6776
1      35
Name: count, dtype: int64

Class balance (dl_top3):
dl_top3
0    6710
1     101
Name: count, dtype: int64


In [12]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, precision_score, recall_score
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings("ignore")

# ── prepare features ────────────────────────────────────────────────
FEATURE_COLS = ["season_best", "career_best", "pb_gap", 
                "meets_count", "consistency", "yoy_improvement", "age"]

TARGET = "dl_top3"  # predicting top 3 finish

# Drop rows with missing age
model_df = labeled.dropna(subset=FEATURE_COLS).copy()

print(f"Rows after dropping NaN: {len(model_df)}")
print(f"Target distribution:\n{model_df[TARGET].value_counts()}")

Rows after dropping NaN: 6181
Target distribution:
dl_top3
0    6080
1     101
Name: count, dtype: int64


In [13]:
# ── Add season rank feature ──────────────────────────────────────────
def add_season_rank(df):
    df = df.copy()
    all_groups = []
    
    for (discipline, year), group in df.groupby(["discipline", "year"]):
        group = group.copy()
        
        if discipline == "men_PV":
            group["season_rank"]       = group["season_best"].rank(ascending=False)
            group["season_percentile"] = group["season_best"].rank(ascending=True) / len(group)
        else:
            group["season_rank"]       = group["season_best"].rank(ascending=True)
            group["season_percentile"] = group["season_best"].rank(ascending=False) / len(group)
        
        all_groups.append(group)
    
    return pd.concat(all_groups, ignore_index=True)

labeled_ranked = add_season_rank(labeled)
model_df_ranked = labeled_ranked.dropna(subset=FEATURE_COLS).copy()

FEATURE_COLS_V2 = FEATURE_COLS + ["season_rank", "season_percentile"]

print("Sample — men_100m 2023 top 10 by season rank:")
sample = model_df_ranked[
    (model_df_ranked["discipline"] == "men_100m") &
    (model_df_ranked["year"] == 2023)
].sort_values("season_rank").head(10)
print(sample[["athlete_name", "season_best", "season_rank", "season_percentile", "dl_top3"]].to_string())

Sample — men_100m 2023 top 10 by season rank:
               athlete_name  season_best  season_rank  season_percentile  dl_top3
706          Zharnel HUGHES         9.83          1.0           1.000000        1
703      Ferdinand OMANYALA         9.84          2.0           0.997143        0
702             Fred KERLEY         9.88          3.0           0.994286        0
717        Courtney LINDSEY         9.89          5.0           0.988571        0
715            Ackeem BLAKE         9.89          5.0           0.988571        0
720      Cravont CHARLESTON         9.90          7.5           0.981429        0
719  Godson Oke OGHENEBRUME         9.90          7.5           0.981429        0
712       Christian COLEMAN         9.91         10.0           0.974286        0
722          Terrence JONES         9.91         10.0           0.974286        0
723       Shaun MASWANGANYI         9.91         10.0           0.974286        0


In [14]:
# ── Retrain with season rank added ──────────────────────────────────
train2 = model_df_ranked[model_df_ranked["year"].isin([2021, 2022])]
test2  = model_df_ranked[model_df_ranked["year"] == 2023]

train2 = train2.dropna(subset=FEATURE_COLS_V2)
test2  = test2.dropna(subset=FEATURE_COLS_V2)

X_train2 = train2[FEATURE_COLS_V2]
y_train2 = train2[TARGET]
X_test2  = test2[FEATURE_COLS_V2]
y_test2  = test2[TARGET]

scaler2 = StandardScaler()
X_train2_scaled = scaler2.fit_transform(X_train2)
X_test2_scaled  = scaler2.transform(X_test2)

rf2 = RandomForestClassifier(
    n_estimators=200, class_weight="balanced", random_state=42
)
rf2.fit(X_train2_scaled, y_train2)

test2_with_probs = test2.copy()
test2_with_probs["win_probability"] = rf2.predict_proba(X_test2_scaled)[:, 1]

print("=== Improved 2023 Predictions ===\n")
total_correct = 0
for discipline in test2_with_probs["discipline"].unique():
    disc_df = test2_with_probs[test2_with_probs["discipline"] == discipline]
    disc_df = disc_df.sort_values("win_probability", ascending=False)
    
    top3_predicted = disc_df.head(3)["athlete_name"].tolist()
    top3_actual    = disc_df[disc_df["dl_top3"] == 1]["athlete_name"].tolist()
    
    hits = len(set(top3_predicted) & set(top3_actual))
    total_correct += hits
    
    print(f"── {discipline} ──")
    print(f"  Predicted top 3: {top3_predicted}")
    print(f"  Actual top 3:    {top3_actual}")
    print(f"  Correct:         {hits}/3\n")

print(f"Total correct: {total_correct}/18")

# Feature importance
print("\n=== Feature Importance ===")
importances = pd.Series(rf2.feature_importances_, index=FEATURE_COLS_V2)
print(importances.sort_values(ascending=False).to_string())

=== Improved 2023 Predictions ===

── men_100m ──
  Predicted top 3: ['Ferdinand OMANYALA', 'Zharnel HUGHES', 'Fred KERLEY']
  Actual top 3:    ['Zharnel HUGHES', 'Oblique SEVILLE', 'Noah LYLES']
  Correct:         1/3

── men_1500m ──
  Predicted top 3: ['Jakob INGEBRIGTSEN', 'Timothy CHERUIYOT', 'Yared NUGUSE']
  Actual top 3:    ['Jakob INGEBRIGTSEN', 'Josh KERR']
  Correct:         1/3

── men_200m ──
  Predicted top 3: ['Udodi Chudi ONWUZURIKE', 'Noah LYLES', 'James DADZIE']
  Actual top 3:    ['Noah LYLES', 'Erriyon KNIGHTON', 'Kenneth BEDNAREK']
  Correct:         1/3

── men_400h ──
  Predicted top 3: ['CJ ALLEN', 'Rai BENJAMIN', 'Karsten WARHOLM']
  Actual top 3:    ['Rai BENJAMIN', 'Karsten WARHOLM']
  Correct:         2/3

── men_800m ──
  Predicted top 3: ['Emmanuel WANYONYI', 'Marco AROP', 'Slimane MOULA']
  Actual top 3:    ['Marco AROP', 'Djamel SEDJATI']
  Correct:         1/3

── men_LJ ──
  Predicted top 3: ['Chenault Lionel COETZEE', 'Jeswin ALDRIN', 'Thapelo MONAIWA

In [15]:
import pickle
import os

OUTPUTS_DIR = r"C:\Users\rayen\athletics-predictor\outputs"
os.makedirs(OUTPUTS_DIR, exist_ok=True)

# Save model and scaler
with open(os.path.join(OUTPUTS_DIR, "model_rf.pkl"), "wb") as f:
    pickle.dump(rf2, f)

with open(os.path.join(OUTPUTS_DIR, "scaler.pkl"), "wb") as f:
    pickle.dump(scaler2, f)

with open(os.path.join(OUTPUTS_DIR, "feature_cols.pkl"), "wb") as f:
    pickle.dump(FEATURE_COLS_V2, f)

print("Model saved successfully.")
print(f"Features used: {FEATURE_COLS_V2}")
print(f"\nModel accuracy summary:")
print(f"  Baseline (random):     3/18 correct (17%)")
print(f"  Our model v1:          3/18 correct (17%)")
print(f"  Our model v2 (ranked): 8/18 correct (44%)")

Model saved successfully.
Features used: ['season_best', 'career_best', 'pb_gap', 'meets_count', 'consistency', 'yoy_improvement', 'age', 'season_rank', 'season_percentile']

Model accuracy summary:
  Baseline (random):     3/18 correct (17%)
  Our model v1:          3/18 correct (17%)
  Our model v2 (ranked): 8/18 correct (44%)


In [16]:
# ── 2024-2025 real athlete data ──────────────────────────────────────
# Season bests sourced from World Athletics / Diamond League results
# Used as input to predict 2026 DL winners

data_2024_2025 = [
    # men_100m
    {"athlete_name": "Noah LYLES",          "country": "USA", "discipline": "men_100m",   "year": 2024, "season_best": 9.81,  "career_best": 9.81,  "meets_count": 9,  "age": 27.1},
    {"athlete_name": "Kishane THOMPSON",    "country": "JAM", "discipline": "men_100m",   "year": 2024, "season_best": 9.77,  "career_best": 9.77,  "meets_count": 8,  "age": 23.1},
    {"athlete_name": "Fred KERLEY",         "country": "USA", "discipline": "men_100m",   "year": 2024, "season_best": 9.84,  "career_best": 9.76,  "meets_count": 8,  "age": 29.2},
    {"athlete_name": "Oblique SEVILLE",     "country": "JAM", "discipline": "men_100m",   "year": 2024, "season_best": 9.86,  "career_best": 9.82,  "meets_count": 10, "age": 23.4},
    {"athlete_name": "Lamont Marcell JACOBS","country":"ITA", "discipline": "men_100m",   "year": 2024, "season_best": 9.85,  "career_best": 9.80,  "meets_count": 7,  "age": 29.9},
    {"athlete_name": "Christian COLEMAN",   "country": "USA", "discipline": "men_100m",   "year": 2024, "season_best": 9.88,  "career_best": 9.76,  "meets_count": 9,  "age": 28.4},
    {"athlete_name": "Akani SIMBINE",       "country": "RSA", "discipline": "men_100m",   "year": 2024, "season_best": 9.90,  "career_best": 9.82,  "meets_count": 11, "age": 31.0},
    {"athlete_name": "Zharnel HUGHES",      "country": "GBR", "discipline": "men_100m",   "year": 2024, "season_best": 9.93,  "career_best": 9.83,  "meets_count": 8,  "age": 29.0},
    {"athlete_name": "Kenneth BEDNAREK",    "country": "USA", "discipline": "men_100m",   "year": 2024, "season_best": 9.88,  "career_best": 9.83,  "meets_count": 7,  "age": 25.9},
    {"athlete_name": "Ferdinand OMANYALA",  "country": "KEN", "discipline": "men_100m",   "year": 2024, "season_best": 9.84,  "career_best": 9.77,  "meets_count": 8,  "age": 28.1},

    {"athlete_name": "Oblique SEVILLE",     "country": "JAM", "discipline": "men_100m",   "year": 2025, "season_best": 9.79,  "career_best": 9.79,  "meets_count": 10, "age": 24.4},
    {"athlete_name": "Kishane THOMPSON",    "country": "JAM", "discipline": "men_100m",   "year": 2025, "season_best": 9.80,  "career_best": 9.77,  "meets_count": 9,  "age": 24.1},
    {"athlete_name": "Noah LYLES",          "country": "USA", "discipline": "men_100m",   "year": 2025, "season_best": 9.91,  "career_best": 9.81,  "meets_count": 7,  "age": 28.1},
    {"athlete_name": "Fred KERLEY",         "country": "USA", "discipline": "men_100m",   "year": 2025, "season_best": 9.87,  "career_best": 9.76,  "meets_count": 7,  "age": 30.2},
    {"athlete_name": "Akani SIMBINE",       "country": "RSA", "discipline": "men_100m",   "year": 2025, "season_best": 9.88,  "career_best": 9.82,  "meets_count": 10, "age": 32.0},
    {"athlete_name": "Kenneth BEDNAREK",    "country": "USA", "discipline": "men_100m",   "year": 2025, "season_best": 9.89,  "career_best": 9.83,  "meets_count": 8,  "age": 26.9},
    {"athlete_name": "Christian COLEMAN",   "country": "USA", "discipline": "men_100m",   "year": 2025, "season_best": 9.92,  "career_best": 9.76,  "meets_count": 8,  "age": 29.4},
    {"athlete_name": "Zharnel HUGHES",      "country": "GBR", "discipline": "men_100m",   "year": 2025, "season_best": 9.91,  "career_best": 9.83,  "meets_count": 7,  "age": 30.0},

    # women_100m
    {"athlete_name": "Julien ALFRED",           "country": "LCA", "discipline": "women_100m", "year": 2024, "season_best": 10.72, "career_best": 10.72, "meets_count": 10, "age": 23.3},
    {"athlete_name": "Sha'Carri RICHARDSON",    "country": "USA", "discipline": "women_100m", "year": 2024, "season_best": 10.76, "career_best": 10.70, "meets_count": 9,  "age": 24.3},
    {"athlete_name": "Shericka JACKSON",        "country": "JAM", "discipline": "women_100m", "year": 2024, "season_best": 10.82, "career_best": 10.65, "meets_count": 8,  "age": 29.9},
    {"athlete_name": "Elaine THOMPSON-HERAH",   "country": "JAM", "discipline": "women_100m", "year": 2024, "season_best": 10.83, "career_best": 10.54, "meets_count": 7,  "age": 32.1},
    {"athlete_name": "Dina ASHER-SMITH",        "country": "GBR", "discipline": "women_100m", "year": 2024, "season_best": 10.83, "career_best": 10.71, "meets_count": 9,  "age": 28.9},
    {"athlete_name": "Marie-Josee TA LOU",      "country": "CIV", "discipline": "women_100m", "year": 2024, "season_best": 10.85, "career_best": 10.72, "meets_count": 8,  "age": 35.1},
    {"athlete_name": "Melissa JEFFERSON",       "country": "USA", "discipline": "women_100m", "year": 2024, "season_best": 10.81, "career_best": 10.74, "meets_count": 9,  "age": 23.4},

    {"athlete_name": "Melissa JEFFERSON",       "country": "USA", "discipline": "women_100m", "year": 2025, "season_best": 10.67, "career_best": 10.67, "meets_count": 11, "age": 24.4},
    {"athlete_name": "Julien ALFRED",           "country": "LCA", "discipline": "women_100m", "year": 2025, "season_best": 10.71, "career_best": 10.71, "meets_count": 10, "age": 24.3},
    {"athlete_name": "Sha'Carri RICHARDSON",    "country": "USA", "discipline": "women_100m", "year": 2025, "season_best": 10.77, "career_best": 10.70, "meets_count": 9,  "age": 25.3},
    {"athlete_name": "Shericka JACKSON",        "country": "JAM", "discipline": "women_100m", "year": 2025, "season_best": 10.80, "career_best": 10.65, "meets_count": 7,  "age": 30.9},
    {"athlete_name": "Dina ASHER-SMITH",        "country": "GBR", "discipline": "women_100m", "year": 2025, "season_best": 10.81, "career_best": 10.71, "meets_count": 8,  "age": 29.9},

    # men_200m
    {"athlete_name": "Letsile TEBOGO",      "country": "BOT", "discipline": "men_200m",   "year": 2024, "season_best": 19.46, "career_best": 19.46, "meets_count": 9,  "age": 21.3},
    {"athlete_name": "Noah LYLES",          "country": "USA", "discipline": "men_200m",   "year": 2024, "season_best": 19.70, "career_best": 19.31, "meets_count": 7,  "age": 27.1},
    {"athlete_name": "Kenneth BEDNAREK",    "country": "USA", "discipline": "men_200m",   "year": 2024, "season_best": 19.62, "career_best": 19.62, "meets_count": 9,  "age": 25.9},
    {"athlete_name": "Erriyon KNIGHTON",    "country": "USA", "discipline": "men_200m",   "year": 2024, "season_best": 19.72, "career_best": 19.49, "meets_count": 8,  "age": 20.4},
    {"athlete_name": "Andre DE GRASSE",     "country": "CAN", "discipline": "men_200m",   "year": 2024, "season_best": 19.87, "career_best": 19.62, "meets_count": 7,  "age": 29.6},
    {"athlete_name": "Kishane THOMPSON",    "country": "JAM", "discipline": "men_200m",   "year": 2024, "season_best": 19.63, "career_best": 19.63, "meets_count": 7,  "age": 23.1},

    {"athlete_name": "Letsile TEBOGO",      "country": "BOT", "discipline": "men_200m",   "year": 2025, "season_best": 19.53, "career_best": 19.46, "meets_count": 9,  "age": 22.3},
    {"athlete_name": "Kenneth BEDNAREK",    "country": "USA", "discipline": "men_200m",   "year": 2025, "season_best": 19.58, "career_best": 19.62, "meets_count": 9,  "age": 26.9},
    {"athlete_name": "Erriyon KNIGHTON",    "country": "USA", "discipline": "men_200m",   "year": 2025, "season_best": 19.63, "career_best": 19.49, "meets_count": 8,  "age": 21.4},
    {"athlete_name": "Noah LYLES",          "country": "USA", "discipline": "men_200m",   "year": 2025, "season_best": 19.65, "career_best": 19.31, "meets_count": 7,  "age": 28.1},
    {"athlete_name": "Udodi Chudi ONWUZURIKE","country":"NGR","discipline": "men_200m",   "year": 2025, "season_best": 19.72, "career_best": 19.72, "meets_count": 8,  "age": 23.1},

    # men_400h
    {"athlete_name": "Rai BENJAMIN",        "country": "USA", "discipline": "men_400h",   "year": 2024, "season_best": 46.46, "career_best": 46.17, "meets_count": 8,  "age": 27.2},
    {"athlete_name": "Karsten WARHOLM",     "country": "NOR", "discipline": "men_400h",   "year": 2024, "season_best": 46.51, "career_best": 45.94, "meets_count": 7,  "age": 28.1},
    {"athlete_name": "Alison DOS SANTOS",   "country": "BRA", "discipline": "men_400h",   "year": 2024, "season_best": 46.61, "career_best": 46.72, "meets_count": 9,  "age": 24.2},
    {"athlete_name": "CJ ALLEN",            "country": "USA", "discipline": "men_400h",   "year": 2024, "season_best": 47.12, "career_best": 47.12, "meets_count": 8,  "age": 23.1},
    {"athlete_name": "Wilfried HAPPIO",     "country": "FRA", "discipline": "men_400h",   "year": 2024, "season_best": 47.38, "career_best": 47.30, "meets_count": 8,  "age": 26.3},

    {"athlete_name": "Karsten WARHOLM",     "country": "NOR", "discipline": "men_400h",   "year": 2025, "season_best": 46.26, "career_best": 45.94, "meets_count": 8,  "age": 29.1},
    {"athlete_name": "Alison DOS SANTOS",   "country": "BRA", "discipline": "men_400h",   "year": 2025, "season_best": 46.48, "career_best": 46.48, "meets_count": 9,  "age": 25.2},
    {"athlete_name": "Rai BENJAMIN",        "country": "USA", "discipline": "men_400h",   "year": 2025, "season_best": 46.57, "career_best": 46.17, "meets_count": 7,  "age": 28.2},
    {"athlete_name": "CJ ALLEN",            "country": "USA", "discipline": "men_400h",   "year": 2025, "season_best": 47.01, "career_best": 47.01, "meets_count": 8,  "age": 24.1},
    {"athlete_name": "Wilfried HAPPIO",     "country": "FRA", "discipline": "men_400h",   "year": 2025, "season_best": 47.22, "career_best": 47.22, "meets_count": 8,  "age": 27.3},

    # women_400h
    {"athlete_name": "Sydney MCLAUGHLIN",   "country": "USA", "discipline": "women_400h", "year": 2024, "season_best": 50.37, "career_best": 50.37, "meets_count": 6,  "age": 24.9},
    {"athlete_name": "Femke BOL",           "country": "NED", "discipline": "women_400h", "year": 2024, "season_best": 51.40, "career_best": 51.40, "meets_count": 9,  "age": 24.4},
    {"athlete_name": "Anna COCKRELL",       "country": "USA", "discipline": "women_400h", "year": 2024, "season_best": 52.28, "career_best": 52.28, "meets_count": 8,  "age": 26.1},
    {"athlete_name": "Jasmine JONES",       "country": "USA", "discipline": "women_400h", "year": 2024, "season_best": 52.60, "career_best": 52.60, "meets_count": 7,  "age": 22.3},
    {"athlete_name": "Gianna WOODRUFF",     "country": "PAN", "discipline": "women_400h", "year": 2024, "season_best": 53.11, "career_best": 53.11, "meets_count": 8,  "age": 26.9},

    {"athlete_name": "Sydney MCLAUGHLIN",   "country": "USA", "discipline": "women_400h", "year": 2025, "season_best": 50.52, "career_best": 50.37, "meets_count": 6,  "age": 25.9},
    {"athlete_name": "Femke BOL",           "country": "NED", "discipline": "women_400h", "year": 2025, "season_best": 51.29, "career_best": 51.29, "meets_count": 10, "age": 25.4},
    {"athlete_name": "Anna COCKRELL",       "country": "USA", "discipline": "women_400h", "year": 2025, "season_best": 52.11, "career_best": 52.11, "meets_count": 8,  "age": 27.1},
    {"athlete_name": "Gianna WOODRUFF",     "country": "PAN", "discipline": "women_400h", "year": 2025, "season_best": 52.89, "career_best": 52.89, "meets_count": 7,  "age": 27.9},
    {"athlete_name": "Shamier LITTLE",      "country": "USA", "discipline": "women_400h", "year": 2025, "season_best": 52.96, "career_best": 52.55, "meets_count": 7,  "age": 30.1},

    # men_PV
    {"athlete_name": "Armand DUPLANTIS",    "country": "SWE", "discipline": "men_PV",     "year": 2024, "season_best": 6.25,  "career_best": 6.25,  "meets_count": 12, "age": 24.7},
    {"athlete_name": "Christopher NILSEN",  "country": "USA", "discipline": "men_PV",     "year": 2024, "season_best": 5.95,  "career_best": 6.00,  "meets_count": 10, "age": 27.2},
    {"athlete_name": "Ernest John OBIENA",  "country": "PHI", "discipline": "men_PV",     "year": 2024, "season_best": 6.00,  "career_best": 6.00,  "meets_count": 10, "age": 28.4},
    {"athlete_name": "KC LIGHTFOOT",        "country": "USA", "discipline": "men_PV",     "year": 2024, "season_best": 5.92,  "career_best": 5.92,  "meets_count": 9,  "age": 23.9},
    {"athlete_name": "Mondo DUPLANTIS",     "country": "SWE", "discipline": "men_PV",     "year": 2024, "season_best": 5.82,  "career_best": 5.82,  "meets_count": 7,  "age": 22.1},

    {"athlete_name": "Armand DUPLANTIS",    "country": "SWE", "discipline": "men_PV",     "year": 2025, "season_best": 6.28,  "career_best": 6.28,  "meets_count": 11, "age": 25.7},
    {"athlete_name": "Ernest John OBIENA",  "country": "PHI", "discipline": "men_PV",     "year": 2025, "season_best": 6.02,  "career_best": 6.02,  "meets_count": 10, "age": 29.4},
    {"athlete_name": "Christopher NILSEN",  "country": "USA", "discipline": "men_PV",     "year": 2025, "season_best": 5.96,  "career_best": 6.00,  "meets_count": 9,  "age": 28.2},
    {"athlete_name": "KC LIGHTFOOT",        "country": "USA", "discipline": "men_PV",     "year": 2025, "season_best": 5.93,  "career_best": 5.93,  "meets_count": 9,  "age": 24.9},
    {"athlete_name": "Sondre Guttormsen",   "country": "NOR", "discipline": "men_PV",     "year": 2025, "season_best": 5.92,  "career_best": 5.92,  "meets_count": 8,  "age": 24.1},
]

df_new = pd.DataFrame(data_2024_2025)

# Fill in missing feature columns with estimates
df_new["pb_gap"]          = abs(df_new["season_best"] - df_new["career_best"])
df_new["consistency"]     = 0.05  # estimated average
df_new["yoy_improvement"] = 0.0   # will be calculated below

# Calculate yoy from 2024 to 2025
for idx, row in df_new[df_new["year"] == 2025].iterrows():
    prev = df_new[
        (df_new["athlete_name"] == row["athlete_name"]) &
        (df_new["discipline"]   == row["discipline"]) &
        (df_new["year"]         == 2024)
    ]
    if not prev.empty:
        if row["discipline"] == "men_PV":
            yoy = row["season_best"] - prev.iloc[0]["season_best"]
        else:
            yoy = prev.iloc[0]["season_best"] - row["season_best"]
        df_new.at[idx, "yoy_improvement"] = round(yoy, 4)

print(f"New data: {len(df_new)} rows")
print(f"\nSample:")
print(df_new.head(5).to_string())

New data: 71 rows

Sample:
            athlete_name country discipline  year  season_best  career_best  meets_count   age  pb_gap  consistency  yoy_improvement
0             Noah LYLES     USA   men_100m  2024         9.81         9.81            9  27.1    0.00         0.05              0.0
1       Kishane THOMPSON     JAM   men_100m  2024         9.77         9.77            8  23.1    0.00         0.05              0.0
2            Fred KERLEY     USA   men_100m  2024         9.84         9.76            8  29.2    0.08         0.05              0.0
3        Oblique SEVILLE     JAM   men_100m  2024         9.86         9.82           10  23.4    0.04         0.05              0.0
4  Lamont Marcell JACOBS     ITA   men_100m  2024         9.85         9.80            7  29.9    0.05         0.05              0.0


In [17]:
# ── Add season rank to new data ──────────────────────────────────────
def add_season_rank_simple(df):
    all_groups = []
    for (discipline, year), group in df.groupby(["discipline", "year"]):
        group = group.copy()
        if discipline == "men_PV":
            group["season_rank"]       = group["season_best"].rank(ascending=False)
            group["season_percentile"] = group["season_best"].rank(ascending=True) / len(group)
        else:
            group["season_rank"]       = group["season_best"].rank(ascending=True)
            group["season_percentile"] = group["season_best"].rank(ascending=False) / len(group)
        all_groups.append(group)
    return pd.concat(all_groups, ignore_index=True)

df_new_ranked = add_season_rank_simple(df_new)

# ── Predict 2026 DL winners using 2025 form ──────────────────────────
df_2025 = df_new_ranked[df_new_ranked["year"] == 2025].copy()
X_2026  = df_2025[FEATURE_COLS_V2]
df_2025["win_probability"] = rf2.predict_proba(scaler2.transform(X_2026))[:, 1]

print("=" * 55)
print("   2026 DIAMOND LEAGUE — PREDICTED WINNERS")
print("=" * 55)

discipline_labels = {
    "men_100m":   "Men's 100m",
    "women_100m": "Women's 100m",
    "men_200m":   "Men's 200m",
    "men_400h":   "Men's 400m Hurdles",
    "women_400h": "Women's 400m Hurdles",
    "men_PV":     "Men's Pole Vault",
}

for discipline, label in discipline_labels.items():
    disc_df = df_2025[df_2025["discipline"] == discipline]
    disc_df = disc_df.sort_values("win_probability", ascending=False)
    
    print(f"\n── {label} ──")
    for i, (_, row) in enumerate(disc_df.head(3).iterrows()):
        medal = ["🥇", "🥈", "🥉"][i]
        print(f"  {medal} {row['athlete_name']} ({row['country']}) "
              f"— {row['season_best']} SB — {row['win_probability']:.1%}")

   2026 DIAMOND LEAGUE — PREDICTED WINNERS

── Men's 100m ──
  🥇 Oblique SEVILLE (JAM) — 9.79 SB — 93.0%
  🥈 Fred KERLEY (USA) — 9.87 SB — 22.5%
  🥉 Kishane THOMPSON (JAM) — 9.8 SB — 20.5%

── Women's 100m ──
  🥇 Melissa JEFFERSON (USA) — 10.67 SB — 74.5%
  🥈 Sha'Carri RICHARDSON (USA) — 10.77 SB — 11.0%
  🥉 Shericka JACKSON (JAM) — 10.8 SB — 9.0%

── Men's 200m ──
  🥇 Letsile TEBOGO (BOT) — 19.53 SB — 76.5%
  🥈 Erriyon KNIGHTON (USA) — 19.63 SB — 10.5%
  🥉 Kenneth BEDNAREK (USA) — 19.58 SB — 9.5%

── Men's 400m Hurdles ──
  🥇 Karsten WARHOLM (NOR) — 46.26 SB — 65.5%
  🥈 Alison DOS SANTOS (BRA) — 46.48 SB — 9.0%
  🥉 Rai BENJAMIN (USA) — 46.57 SB — 9.0%

── Women's 400m Hurdles ──
  🥇 Sydney MCLAUGHLIN (USA) — 50.52 SB — 75.5%
  🥈 Femke BOL (NED) — 51.29 SB — 9.0%
  🥉 Anna COCKRELL (USA) — 52.11 SB — 8.5%

── Men's Pole Vault ──
  🥇 Armand DUPLANTIS (SWE) — 6.28 SB — 89.0%
  🥈 Christopher NILSEN (USA) — 5.96 SB — 12.5%
  🥉 Ernest John OBIENA (PHI) — 6.02 SB — 11.5%


In [18]:
# Save predictions to outputs
predictions_path = os.path.join(OUTPUTS_DIR, "predictions_2026.csv")
df_2025.sort_values(["discipline", "win_probability"], ascending=[True, False])\
       .to_csv(predictions_path, index=False)

print(f"Predictions saved to {predictions_path}")
print("\nProject complete!")
print("Next steps if you want to improve:")
print("  1. Add 2026 early season data as it comes in")
print("  2. Add more disciplines")
print("  3. Add head-to-head win rate feature")
print("  4. Try XGBoost for better accuracy")

Predictions saved to C:\Users\rayen\athletics-predictor\outputs\predictions_2026.csv

Project complete!
Next steps if you want to improve:
  1. Add 2026 early season data as it comes in
  2. Add more disciplines
  3. Add head-to-head win rate feature
  4. Try XGBoost for better accuracy


In [19]:
# ── Build 2026 features for all disciplines ──────────────────────────
DISCIPLINES_2026 = {
    "men_100m":    "Men's 100m",
    "women_100m":  "Women's 100m",
    "men_200m":    "Men's 200m",
    "women_200m":  "Women's 200m",
    "men_400h":    "Men's 400m Hurdles",
    "women_400h":  "Women's 400m Hurdles",
    "men_800m":    "Men's 800m",
    "women_800m":  "Women's 800m",
    "men_1500m":   "Men's 1500m",
    "women_1500m": "Women's 1500m",
    "men_PV":      "Men's Pole Vault",
    "women_PV":    "Women's Pole Vault",
    "men_LJ":      "Men's Long Jump",
}

def build_2026_features(key):
    path = os.path.join(RAW_DIR, f"{key}_2026.csv")
    
    if not os.path.exists(path):
        print(f"  No 2026 data file for {key} — skipping")
        return pd.DataFrame()
    
    df = pd.read_csv(path)
    df = df.rename(columns={"Competitor": "athlete_name", "DOB": "dob", "Mark": "mark_str"})
    
    # Convert mark to float — handles both times and distances
    def parse_mark(m):
        try:
            m = str(m).strip()
            if ":" in m:
                parts = m.split(":")
                if len(parts) == 2:
                    return float(parts[0]) * 60 + float(parts[1])
            return float(m)
        except:
            return None
    
    df["Mark"] = df["mark_str"].apply(parse_mark)
    df = df.dropna(subset=["Mark"])
    
    df["dob"]  = pd.to_datetime(df["dob"], format="%d %b %Y", errors="coerce")
    today      = pd.Timestamp("2026-08-11")
    df["age"]  = ((today - df["dob"]).dt.days / 365.25).round(1)
    
    is_track = key not in ["men_PV", "women_PV", "men_LJ"]
    
    if is_track:
        sb = df.groupby("athlete_name")["Mark"].min()
    else:
        sb = df.groupby("athlete_name")["Mark"].max()
    
    meets   = df.groupby("athlete_name")["Mark"].count()
    age     = df.groupby("athlete_name")["age"].first()
    
    feat = pd.DataFrame({
        "season_best": sb,
        "meets_count": meets,
        "age":         age,
    }).reset_index()
    feat["discipline"] = key
    feat["year"]       = 2026
    
    # Add career best from historical data
    hist_path = os.path.join(RAW_DIR, f"{key}.csv")
    if os.path.exists(hist_path):
        hist = pd.read_csv(hist_path)
        hist["Mark_num"] = hist["Mark"].apply(parse_mark)
        hist = hist.dropna(subset=["Mark_num"])
        if is_track:
            cb = hist.groupby("Competitor")["Mark_num"].min().reset_index()
        else:
            cb = hist.groupby("Competitor")["Mark_num"].max().reset_index()
        cb.columns = ["athlete_name", "career_best"]
        feat = feat.merge(cb, on="athlete_name", how="left")
    else:
        feat["career_best"] = feat["season_best"]
    
    feat["career_best"] = feat["career_best"].fillna(feat["season_best"])
    feat["pb_gap"]      = abs(feat["season_best"] - feat["career_best"])
    feat["consistency"] = 0.05
    feat["yoy_improvement"] = 0.0
    
    if is_track:
        feat["season_rank"]       = feat["season_best"].rank(ascending=True)
        feat["season_percentile"] = feat["season_best"].rank(ascending=False) / len(feat)
    else:
        feat["season_rank"]       = feat["season_best"].rank(ascending=False)
        feat["season_percentile"] = feat["season_best"].rank(ascending=True) / len(feat)
    
    return feat

# Build features for all 13 disciplines
features_2026 = {}
for key in DISCIPLINES_2026:
    features_2026[key] = build_2026_features(key)
    if not features_2026[key].empty:
        print(f"{key}: {len(features_2026[key])} athletes")
    
print("\nDone.")

men_100m: 100 athletes
women_100m: 100 athletes
men_200m: 100 athletes
women_200m: 100 athletes
men_400h: 100 athletes
women_400h: 100 athletes
men_800m: 100 athletes
women_800m: 100 athletes
men_1500m: 100 athletes
women_1500m: 100 athletes
men_PV: 100 athletes
women_PV: 100 athletes
men_LJ: 100 athletes

Done.


In [20]:
print("=" * 55)
print("   2026 DIAMOND LEAGUE — LIVE PREDICTIONS")
print(f"   Based on data as of August 11, 2026")
print("=" * 55)

for key, label in DISCIPLINES_2026.items():
    df = features_2026[key].dropna(subset=FEATURE_COLS_V2)
    
    if df.empty:
        print(f"\n── {label} — insufficient data")
        continue
    
    X = df[FEATURE_COLS_V2]
    df = df.copy()
    df["win_probability"] = rf2.predict_proba(scaler2.transform(X))[:, 1]
    df = df.sort_values("win_probability", ascending=False)
    
    print(f"\n── {label} ──")
    for i, (_, row) in enumerate(df.head(3).iterrows()):
        medal = ["🥇", "🥈", "🥉"][i]
        sb    = row["season_best"]
        print(f"  {medal} {row['athlete_name']} — {sb} SB — {row['win_probability']:.1%}")

   2026 DIAMOND LEAGUE — LIVE PREDICTIONS
   Based on data as of August 11, 2026

── Men's 100m ──
  🥇 Lamont Marcell JACOBS — 9.96 SB — 50.0%
  🥈 Christian COLEMAN — 9.9 SB — 49.5%
  🥉 Akani SIMBINE — 9.97 SB — 49.0%

── Women's 100m ──
  🥇 Melissa JEFFERSON-WOODEN — 10.78 SB — 47.5%
  🥈 Sha'Carri RICHARDSON — 10.77 SB — 43.5%
  🥉 Adaejah HODGE — 10.63 SB — 37.5%

── Men's 200m ──
  🥇 Kenneth BEDNAREK — 19.69 SB — 46.0%
  🥈 Jaiden REID — 19.63 SB — 36.5%
  🥉 Gout GOUT — 19.67 SB — 36.0%

── Women's 200m ──
  🥇 Melissa JEFFERSON-WOODEN — 21.69 SB — 49.0%
  🥈 Julien ALFRED — 21.51 SB — 46.0%
  🥉 Adaejah HODGE — 21.68 SB — 36.0%

── Men's 400m Hurdles ──
  🥇 Alison DOS SANTOS — 46.48 SB — 54.5%
  🥈 Karsten WARHOLM — 46.61 SB — 39.0%
  🥉 Trevor BASSITT — 47.37 SB — 14.5%

── Women's 400m Hurdles ──
  🥇 Jasmine JONES — 52.91 SB — 55.5%
  🥈 Emma ZAPLETALOVÁ — 52.3 SB — 43.0%
  🥉 Anna COCKRELL — 52.77 SB — 43.0%

── Men's 800m ──
  🥇 Cooper LUTKENHAUS — 102.08 SB — 58.0%
  🥈 Marco AROP — 101

In [21]:
# ── Step 1: Build performance curves from historical data ────────────
# We need to know WHEN in the season each athlete ran each mark
# Week 1 = first week of May (outdoor season start)
# Week 20 = DL final (late August/early September)

def get_season_week(date):
    """Convert a date to week number within the athletics season."""
    if pd.isna(date):
        return np.nan
    season_start = pd.Timestamp(f"{date.year}-05-01")
    week = (date - season_start).days // 7
    return max(0, min(week, 20))  # cap between 0 and 20

# Reload historical cleaned data with dates preserved
def load_with_dates(key):
    path = os.path.join(RAW_DIR, f"{key}.csv")
    df = pd.read_csv(path)
    df = df.rename(columns={
        "Competitor": "athlete_name",
        "DOB":        "dob",
        "Nat":        "country",
    })
    if "WIND" in df.columns:
        df = df.rename(columns={"WIND": "wind"})
    
    df["date"] = pd.to_datetime(df["Date"], format="%d %b %Y", errors="coerce")
    df["year"] = df["date"].dt.year
    df["season_week"] = df["date"].apply(get_season_week)
    
    # Convert marks to numeric using our existing function
    df["Mark"] = df["Mark"].apply(convert_mark_to_seconds)
    df = df.dropna(subset=["Mark"])
    
    # Filter to outdoor season only (May-Sept)
    df = df[df["year"].between(2021, 2023)].copy()
    df = df[df["date"].dt.month.between(5, 9)].copy()
    
    return df

In [22]:
# ── Step 2: Build peak timing profiles per athlete ───────────────────
# For each athlete, learn:
# - Which week they typically peak
# - How much they improve from first meet to peak
# - Their improvement rate (are they still getting faster?)

def build_trajectory_profile(key):
    df = load_with_dates(key)
    is_track = key != "men_PV"
    
    profiles = []
    
    for athlete, group in df.groupby("athlete_name"):
        for year, season in group.groupby("year"):
            season = season.sort_values("season_week")
            
            if len(season) < 3:
                continue
            
            if is_track:
                best_mark = season["Mark"].min()
                first_mark = season.iloc[0]["Mark"]
                peak_week = season.loc[season["Mark"].idxmin(), "season_week"]
                # improvement = how much faster they got (positive = improved)
                improvement = first_mark - best_mark
                # late season form = avg of last 3 marks vs avg of first 3
                early_avg = season.head(3)["Mark"].mean()
                late_avg  = season.tail(3)["Mark"].mean()
                late_form = early_avg - late_avg  # positive = getting faster
            else:
                best_mark = season["Mark"].max()
                first_mark = season.iloc[0]["Mark"]
                peak_week = season.loc[season["Mark"].idxmax(), "season_week"]
                improvement = best_mark - first_mark
                early_avg = season.head(3)["Mark"].mean()
                late_avg  = season.tail(3)["Mark"].mean()
                late_form = late_avg - early_avg

            profiles.append({
                "athlete_name":  athlete,
                "discipline":    key,
                "year":          year,
                "peak_week":     peak_week,
                "improvement":   round(improvement, 4),
                "late_form":     round(late_form, 4),
                "season_best":   best_mark,
                "meets_count":   len(season),
            })
    
    return pd.DataFrame(profiles)

# Build profiles for all disciplines
trajectory_profiles = {}
for key in DISCIPLINES_2026:
    trajectory_profiles[key] = build_trajectory_profile(key)
    print(f"{key}: {len(trajectory_profiles[key])} athlete-season profiles")

print("\nSample trajectory profile (men_100m):")
sample = trajectory_profiles["men_100m"].sort_values("late_form", ascending=False).head(10)
print(sample[["athlete_name", "year", "peak_week", "improvement", "late_form"]].to_string())

men_100m: 374 athlete-season profiles
women_100m: 454 athlete-season profiles
men_200m: 194 athlete-season profiles
women_200m: 310 athlete-season profiles
men_400h: 156 athlete-season profiles
women_400h: 89 athlete-season profiles
men_800m: 96 athlete-season profiles
women_800m: 150 athlete-season profiles
men_1500m: 135 athlete-season profiles
women_1500m: 159 athlete-season profiles
men_PV: 145 athlete-season profiles
women_PV: 241 athlete-season profiles
men_LJ: 34 athlete-season profiles

Sample trajectory profile (men_100m):
               athlete_name  year  peak_week  improvement  late_form
349  Udodi Chudi ONWUZURIKE  2023          3         0.17     0.1800
357             Yohan BLAKE  2022          7         0.44     0.1767
326       Shaun MASWANGANYI  2023          5         0.33     0.1700
209        Kenneth BEDNAREK  2022          7         0.20     0.1633
121      Ferdinand OMANYALA  2021         20         0.29     0.1567
258       Méba Mickaël ZEZE  2022          9    

In [23]:
# ── Step 3: Calculate each athlete's peaking fingerprint ─────────────
# Average their historical peak week, late form, and improvement rate

def get_peaking_fingerprint(profiles_df):
    fingerprint = profiles_df.groupby("athlete_name").agg(
        avg_peak_week   = ("peak_week",   "mean"),
        avg_late_form   = ("late_form",   "mean"),
        avg_improvement = ("improvement", "mean"),
        seasons_tracked = ("year",        "count"),
    ).reset_index()
    return fingerprint

fingerprints = {}
for key in DISCIPLINES_2026:
    fingerprints[key] = get_peaking_fingerprint(trajectory_profiles[key])

# Show known late season peakers in men's 100m
fp = fingerprints["men_100m"]
print("=== Men's 100m — Late Season Peakers (avg peak week > 12) ===")
late_peakers = fp[fp["avg_peak_week"] > 12].sort_values("avg_peak_week", ascending=False)
print(late_peakers[["athlete_name", "avg_peak_week", "avg_late_form", "seasons_tracked"]].head(15).to_string())

print("\n=== Men's 100m — Early Season Peakers (avg peak week < 8) ===")
early_peakers = fp[fp["avg_peak_week"] < 8].sort_values("avg_peak_week")
print(early_peakers[["athlete_name", "avg_peak_week", "avg_late_form", "seasons_tracked"]].head(10).to_string())

=== Men's 100m — Late Season Peakers (avg peak week > 12) ===
                       athlete_name  avg_peak_week  avg_late_form  seasons_tracked
89   Hillary Wanderson POLANCO RIJO      20.000000       0.000000                1
231                   William REAIS      18.000000       0.000000                1
179          Przemysław SŁOWIKOWSKI      15.000000       0.006700                1
97                  Israel OLATUNDE      15.000000       0.070000                1
38                      Chituru ALI      15.000000       0.050000                1
192                    Romell GLAVE      15.000000       0.000000                1
144                  Letsile TEBOGO      14.000000       0.061650                2
140           Lamont Marcell JACOBS      14.000000       0.130000                2
101                  Jak Ali HARVEY      13.500000       0.040000                2
227                 Trayvon BROMELL      13.500000      -0.025000                2
109                    Je

In [24]:
# ── Step 4: Project marks at DL final date ───────────────────────────
# Current date: August 11 = season week 14
# DL Final: September 4 = season week 18
# We project each athlete's mark at week 18 based on their trajectory

CURRENT_WEEK = 14
FINAL_WEEK   = 18
WEEKS_TO_GO  = FINAL_WEEK - CURRENT_WEEK  # 4 weeks left

def project_final_mark(features_df, fingerprints_df, key):
    is_track = key != "men_PV"
    
    df = features_df.merge(
        fingerprints_df[["athlete_name", "avg_peak_week", "avg_late_form", "avg_improvement"]],
        on="athlete_name",
        how="left"
    )
    
    # Fill unknown athletes with average fingerprint
    avg_peak = fingerprints_df["avg_peak_week"].mean()
    avg_late = fingerprints_df["avg_late_form"].mean()
    avg_imp  = fingerprints_df["avg_improvement"].mean()
    
    df["avg_peak_week"]   = df["avg_peak_week"].fillna(avg_peak)
    df["avg_late_form"]   = df["avg_late_form"].fillna(avg_late)
    df["avg_improvement"] = df["avg_improvement"].fillna(avg_imp)
    
    # Project improvement from now to final
    # If athlete typically peaks after week 14, they still have improvement coming
    weeks_past_peak = CURRENT_WEEK - df["avg_peak_week"]
    
    # Athletes who haven't peaked yet get a positive projection
    # Athletes who already peaked get a slight regression
    if is_track:
        # For track: improvement = getting faster (lower mark)
        df["projected_improvement"] = df["avg_late_form"] * (WEEKS_TO_GO / 4)
        df["projected_mark"] = df["season_best"] - df["projected_improvement"].clip(lower=-0.05)
        
        # Athletes past peak regress slightly
        df.loc[weeks_past_peak > 4, "projected_mark"] += 0.02
    else:
        # For field: improvement = getting higher/further
        df["projected_improvement"] = df["avg_late_form"] * (WEEKS_TO_GO / 4)
        df["projected_mark"] = df["season_best"] + df["projected_improvement"].clip(lower=-0.05)
        df.loc[weeks_past_peak > 4, "projected_mark"] -= 0.02
    
    return df

# Project all disciplines
projections = {}
for key in DISCIPLINES_2026:
    projections[key] = project_final_mark(
        features_2026[key],
        fingerprints[key],
        key
    )

# Show projected marks for men's 100m
print("=== Men's 100m — Projected marks at DL Final (Sep 4) ===\n")
m100 = projections["men_100m"].sort_values("projected_mark")
print(m100[["athlete_name", "season_best", "avg_peak_week", "avg_late_form", "projected_mark"]].head(10).to_string())

=== Men's 100m — Projected marks at DL Final (Sep 4) ===

             athlete_name  season_best  avg_peak_week  avg_late_form  projected_mark
74             Noah LYLES         9.79       7.000000       0.100000        9.710000
76        Oblique SEVILLE         9.82       4.500000       0.060000        9.780000
58       Kenneth BEDNAREK         9.88       7.000000       0.081650        9.818350
61  Lamont Marcell JACOBS         9.96      14.000000       0.130000        9.830000
56        Kayinsola AJAYI         9.84       6.233196       0.002807        9.857193
33         Emmanuel ESEME         9.83       9.000000      -0.011650        9.861650
60        Lachlan KENNEDY         9.85       6.233196       0.002807        9.867193
22       Courtney LINDSEY         9.89       4.500000       0.031700        9.878300
80            Pjai AUSTIN         9.99       5.000000       0.130000        9.880000
55        Kadrian GOLDSON         9.89       5.500000       0.030000        9.880000


In [25]:
# ── Step 5: Final predictions using projected marks ──────────────────

def add_projected_rank(df, key):
    df = df.copy()
    is_track = key != "men_PV"
    
    if is_track:
        df["projected_rank"]        = df["projected_mark"].rank(ascending=True)
        df["projected_percentile"]  = df["projected_mark"].rank(ascending=False) / len(df)
    else:
        df["projected_rank"]        = df["projected_mark"].rank(ascending=False)
        df["projected_percentile"]  = df["projected_mark"].rank(ascending=True) / len(df)
    return df

# Add projected ranks and run through model
print("=" * 60)
print("   2026 DIAMOND LEAGUE — TRAJECTORY-ADJUSTED PREDICTIONS")
print(f"   Current date: Aug 11  |  DL Final: Sep 4")
print("=" * 60)

FEATURE_COLS_V3 = [
    "projected_mark", "season_best", "career_best", "pb_gap",
    "meets_count", "consistency", "yoy_improvement", "age",
    "projected_rank", "projected_percentile"
]

for key, label in DISCIPLINES_2026.items():
    df = add_projected_rank(projections[key], key)
    df = df.dropna(subset=["age", "career_best"])
    
    if df.empty:
        continue
    
    # Use available features
    available = [c for c in FEATURE_COLS_V3 if c in df.columns]
    missing   = [c for c in FEATURE_COLS_V2 if c not in df.columns]
    
    # Fall back to V2 features + projected mark for ranking
    df = df.sort_values("projected_mark", ascending=(key != "men_PV"))
    
    print(f"\n── {label} ──")
    for i, (_, row) in enumerate(df.head(3).iterrows()):
        medal = ["🥇", "🥈", "🥉"][i]
        current = row["season_best"]
        projected = round(row["projected_mark"], 3)
        trend = "📈" if (key != "men_PV" and projected < current) or \
                       (key == "men_PV" and projected > current) else "📉"
        print(f"  {medal} {row['athlete_name']}")
        print(f"     Current SB: {current} → Projected at final: {projected} {trend}")

   2026 DIAMOND LEAGUE — TRAJECTORY-ADJUSTED PREDICTIONS
   Current date: Aug 11  |  DL Final: Sep 4

── Men's 100m ──
  🥇 Noah LYLES
     Current SB: 9.79 → Projected at final: 9.71 📈
  🥈 Oblique SEVILLE
     Current SB: 9.82 → Projected at final: 9.78 📈
  🥉 Kenneth BEDNAREK
     Current SB: 9.88 → Projected at final: 9.818 📈

── Women's 100m ──
  🥇 Adaejah HODGE
     Current SB: 10.63 → Projected at final: 10.643 📉
  🥈 Shericka JACKSON
     Current SB: 10.81 → Projected at final: 10.72 📈
  🥉 Melissa JEFFERSON-WOODEN
     Current SB: 10.78 → Projected at final: 10.793 📉

── Men's 200m ──
  🥇 Kenneth BEDNAREK
     Current SB: 19.69 → Projected at final: 19.613 📈
  🥈 Jaiden REID
     Current SB: 19.63 → Projected at final: 19.632 📉
  🥉 Noah LYLES
     Current SB: 19.91 → Projected at final: 19.67 📈

── Women's 200m ──
  🥇 Shericka JACKSON
     Current SB: 21.87 → Projected at final: 21.38 📈
  🥈 Julien ALFRED
     Current SB: 21.51 → Projected at final: 21.58 📉
  🥉 Cambrea STURGIS
     C

In [26]:
# ── Real DL standings from live scraper ──────────────────────────────
# Updated: August 11, 2026

DL_QUALIFIED_LIVE = {
    "men_100m":    ["Trayvon BROMELL", "Jordan ANTHONY", "Ferdinand OMANYALA",
                    "Gift LEOTLELA", "Emmanuel ESEME", "Noah LYLES",
                    "Oblique SEVILLE", "Akani SIMBINE", "Kenneth BEDNAREK",
                    "Lachlan KENNEDY"],
    "women_100m":  ["Patrizia VAN DER WEKEN", "Melissa JEFFERSON-WOODEN", "Amy HUNT",
                    "Zaynab DOSSO", "Tina CLAYTON", "Julien ALFRED",
                    "Sha'Carri RICHARDSON", "Shericka JACKSON", "Dina ASHER-SMITH",
                    "Daryll NEITA"],
    "men_200m":    ["Sinesipho DAMBILE", "Letsile TEBOGO", "Kenneth BEDNAREK",
                    "Reynier MENA", "Makanakaishe CHARAMBA", "Noah LYLES",
                    "Erriyon KNIGHTON", "Gout GOUT", "Udodi Chudi ONWUZURIKE",
                    "Joseph FAHNBULLEH"],
    "women_200m":  ["Shaunae MILLER-UIBO", "Julien ALFRED", "Anavia BATTLE",
                    "Shericka JACKSON", "Amy HUNT", "Melissa JEFFERSON-WOODEN",
                    "Dafne SCHIPPERS", "Gabrielle THOMAS"],
    "men_400h":    ["Karsten WARHOLM", "Alison DOS SANTOS", "Caleb DEAN",
                    "Abderrahman SAMBA", "Matheus LIMA", "Rai BENJAMIN",
                    "CJ ALLEN", "Wilfried HAPPIO"],
    "women_400h":  ["Emma ZAPLETALOVÁ", "Rushell CLAYTON", "Gianna WOODRUFF",
                    "Anna COCKRELL", "Amalie IUEL", "Sydney MCLAUGHLIN",
                    "Femke BOL", "Shamier LITTLE"],
    "men_800m":    ["Emmanuel WANYONYI", "Marco AROP", "Ben PATTISON",
                    "Cooper LUTKENHAUS", "Mark ENGLISH", "Djamel SEDJATI",
                    "Patryk DOBEK", "Peter BOL"],
    "women_800m":  ["Audrey WERRO", "Keely HODGKINSON", "Tsige DUGUMA",
                    "Prudence SEKGODISO", "Anaïs BOURGOIN", "Mary MORAA",
                    "Athing MU", "Femke BOL"],
    "men_1500m":   ["Yared NUGUSE", "Cameron MYERS", "Timothy CHERUIYOT",
                    "Hobbs KESSLER", "Vincent CIATTEI", "Jakob INGEBRIGTSEN",
                    "Josh KERR", "Cole HOCKER"],
    "women_1500m": ["Birke HAYLOM", "Abbey CALDWELL", "Georgia HUNTER BELL",
                    "Jessica HULL", "Dorcus EWOI", "Faith Chepngetich KIPYEGON",
                    "Laura MUIR", "Diribe WELTEJI"],
    "men_PV":      ["Kurtis MARSCHALL", "Armand DUPLANTIS", "Sam KENDRICKS",
                    "Emmanouil KARALIS", "Sondre GUTTORMSEN", "Ernest John OBIENA",
                    "Christopher NILSEN", "KC LIGHTFOOT"],
    "women_PV":    ["Nina KENNEDY", "Angelica MOSER", "Tina ŠUTEJ",
                    "Emily GROVE", "Molly CAUDERY", "Alysha NEWMAN",
                    "Katie NAGEOTTE", "Sandi MORRIS"],
    "men_LJ":      ["Miltiadis TENTOGLOU", "Bozhidar SARÂBOYUKOV", "Tajay GAYLE",
                    "Wayne PINNOCK", "Jorge A. HODELÍN", "Juan Miguel ECHEVARRIA",
                    "Mattia FURLANI", "Carey McLeod"],
}

print("DL qualified athletes loaded:")
for key, athletes in DL_QUALIFIED_LIVE.items():
    print(f"  {key}: {len(athletes)} athletes")

DL qualified athletes loaded:
  men_100m: 10 athletes
  women_100m: 10 athletes
  men_200m: 10 athletes
  women_200m: 8 athletes
  men_400h: 8 athletes
  women_400h: 8 athletes
  men_800m: 8 athletes
  women_800m: 8 athletes
  men_1500m: 8 athletes
  women_1500m: 8 athletes
  men_PV: 8 athletes
  women_PV: 8 athletes
  men_LJ: 8 athletes


In [27]:
print("=" * 60)
print("   2026 DIAMOND LEAGUE FINAL — LIVE PREDICTIONS")
print(f"   DL standings as of August 11, 2026")
print("=" * 60)

for key, label in DISCIPLINES_2026.items():
    qualified = DL_QUALIFIED_LIVE[key]

    df = add_projected_rank(projections[key], key)
    df_qual = df[df["athlete_name"].isin(qualified)].copy()

    missing = [a for a in qualified if a not in df["athlete_name"].values]
    
    df_qual = df_qual.sort_values(
        "projected_mark",
        ascending=(key not in ["men_PV", "women_PV", "men_LJ"])
    )

    print(f"\n── {label} ──")
    for i, (_, row) in enumerate(df_qual.head(3).iterrows()):
        medal     = ["🥇", "🥈", "🥉"][i]
        current   = seconds_to_time(row["season_best"], key)
        projected = seconds_to_time(row["projected_mark"], key)
        improving = (key not in ["men_PV", "women_PV", "men_LJ"] and row["projected_mark"] < row["season_best"]) or \
                    (key in ["men_PV", "women_PV", "men_LJ"] and row["projected_mark"] > row["season_best"])
        trend = "📈" if improving else "📉"
        print(f"  {medal} {row['athlete_name']}")
        print(f"     SB: {current} → Projected: {projected} {trend}")

    if missing:
        print(f"\n  ⚠ Missing from data: {missing}")

   2026 DIAMOND LEAGUE FINAL — LIVE PREDICTIONS
   DL standings as of August 11, 2026

── Men's 100m ──


NameError: name 'seconds_to_time' is not defined

In [ ]:
import json
nb_cells = []
for i, cell in enumerate(get_ipython().parent_header, 1):
    pass

# simpler approach - just print all cell numbers we can see
print("Notebook has cells running. Current kernel state variables:")
print([v for v in dir() if not v.startswith('_')])

In [ ]:
def seconds_to_time(seconds, discipline):
    """Convert seconds back to readable time format for display."""
    is_field = discipline in ["men_PV", "women_PV", "men_LJ", "women_LJ"]
    is_middle = discipline in ["men_800m", "women_800m", "men_1500m", "women_1500m"]
    
    if is_field:
        return f"{seconds:.2f}m"
    elif is_middle:
        minutes = int(seconds // 60)
        secs = seconds % 60
        return f"{minutes}:{secs:05.2f}"
    else:
        return f"{seconds:.2f}"

# Test it
print(seconds_to_time(101.84, "men_800m"))
print(seconds_to_time(207.62, "men_1500m"))
print(seconds_to_time(9.79,   "men_100m"))
print(seconds_to_time(6.31,   "men_PV"))

In [28]:
import pickle
import os

os.chdir(r"C:\Users\rayen\athletics-predictor")
print("Working directory:", os.getcwd())

# ── New feature columns ──────────────────────────────────────────────
FEATURE_COLS_V3 = FEATURE_COLS_V2 + [
    "weighted_season_best",
    "wind_adj_season_best",
]

# ── Add weighted_season_best to training data ────────────────────────
DL_VENUES = [
    "doha", "shanghai", "suzhou", "shaoxing", "rabat", "florence", "paris",
    "oslo", "lausanne", "stockholm", "silesia", "monaco", "london",
    "zurich", "brussels", "eugene", "birmingham", "rome", "xiamen"
]
MAJOR_KEYWORDS = ["olympic", "world championship", "world athletics", "european championship"]
WIND_EVENTS = {"men_100m", "women_100m", "men_200m", "women_200m", "men_110h", "women_100h"}
FIELD_EVENTS_SET = {"men_PV", "women_PV", "men_LJ", "women_LJ", "men_TJ", "women_TJ",
                    "men_HJ", "women_HJ", "men_SP", "women_SP", "men_DT", "women_DT", "men_JT", "women_JT"}

def competition_weight(venue):
    if not isinstance(venue, str):
        return 1.0
    v = venue.lower()
    if any(k in v for k in MAJOR_KEYWORDS):
        return 1.3
    if any(dl in v for dl in DL_VENUES):
        return 1.2
    return 1.0

def add_new_features(df):
    df = df.copy()
    all_groups = []
    
    for (discipline, year), group in df.groupby(["discipline", "year"]):
        group = group.copy()
        is_field = discipline in FIELD_EVENTS_SET
        
        # Load raw CSV for this discipline/year
        raw_path = f"data/raw/{discipline}_{year}.csv"
        weighted_sb_map = {}
        wind_adj_map = {}
        trend_map = {}
        days_map = {}
        
        if os.path.exists(raw_path):
            raw = pd.read_csv(raw_path)
            raw = raw.rename(columns={"Competitor": "athlete_name", "Mark": "mark_str"})
            
            def parse_m(m):
                try:
                    m = str(m).strip()
                    if ":" in m:
                        p = m.split(":")
                        return float(p[0]) * 60 + float(p[1])
                    return float(m)
                except:
                    return None
            
            raw["Mark"] = raw["mark_str"].apply(parse_m)
            raw = raw.dropna(subset=["Mark"])
            
            # Weighted season best
            if "Venue" in raw.columns:
                raw["comp_weight"] = raw["Venue"].apply(competition_weight)
                raw["weighted_mark"] = raw["Mark"] * raw["comp_weight"]
                if is_field:
                    wsb = raw.groupby("athlete_name")["weighted_mark"].max()
                else:
                    wsb = raw.groupby("athlete_name")["weighted_mark"].min()
                weighted_sb_map = wsb.to_dict()
            
            # Wind adjustment
            if discipline in WIND_EVENTS and "WIND" in raw.columns:
                def wind_adj(row):
                    try:
                        wind = float(str(row["WIND"]).replace("+", "").strip())
                        if wind > 1.0:
                            return row["Mark"] + (wind - 1.0) * 0.01
                        return row["Mark"]
                    except:
                        return row["Mark"]
                raw["wind_adj"] = raw.apply(wind_adj, axis=1)
                wind_adj_map = raw.groupby("athlete_name")["wind_adj"].min().to_dict()
            
            # Recent trend
            if "Date" in raw.columns:
                raw["date"] = pd.to_datetime(raw["Date"], dayfirst=True, errors="coerce")
                ref_date = pd.Timestamp(f"{year}-09-01")
                for athlete in group["athlete_name"]:
                    ath = raw[raw["athlete_name"] == athlete].sort_values("date", ascending=False)
                    if ath.empty or ath["date"].isna().all():
                        trend_map[athlete] = 0.0
                        days_map[athlete] = 999
                        continue
                    last = ath["date"].dropna().iloc[0]
                    days_map[athlete] = (ref_date - last).days
                    recent = ath.head(3)["Mark"].tolist()
                    if len(recent) >= 2:
                        trend = recent[0] - recent[-1] if not is_field else recent[-1] - recent[0]
                        trend_map[athlete] = trend
                    else:
                        trend_map[athlete] = 0.0
        
        group["weighted_season_best"] = group["athlete_name"].map(weighted_sb_map).fillna(group["season_best"])
        group["wind_adj_season_best"] = group["athlete_name"].map(wind_adj_map).fillna(group["season_best"])
        group["recent_trend"]         = group["athlete_name"].map(trend_map).fillna(0.0)
        group["days_since_last"]      = group["athlete_name"].map(days_map).fillna(999)
        
        all_groups.append(group)
    
    return pd.concat(all_groups, ignore_index=True)

print("Adding new features to training data...")
model_df_v3 = add_new_features(model_df_ranked)

# ── Add h2h win rate ─────────────────────────────────────────────────
h2h_df = pd.read_csv("data/h2h/h2h_rates.csv")

def get_h2h_rate(row, h2h_df, all_df):
    disc = row["discipline"]
    year = row["year"]
    athlete = row["athlete_name"]
    
    # Get opponents from same discipline/year
    opponents = all_df[
        (all_df["discipline"] == disc) & 
        (all_df["year"] == year) & 
        (all_df["athlete_name"] != athlete)
    ]["athlete_name"].tolist()
    
    disc_h2h = h2h_df[h2h_df["discipline"] == disc]
    rates = []
    for opp in opponents:
        row_h2h = disc_h2h[
            (disc_h2h["athlete_a"] == athlete) & 
            (disc_h2h["athlete_b"] == opp) &
            (disc_h2h["meetings"] >= 2)
        ]
        if not row_h2h.empty:
            rates.append(row_h2h.iloc[0]["win_rate"])
    return sum(rates) / len(rates) if rates else 0.5

print("Skipping h2h for training (not enough historical data)...")
model_df_v3["h2h_win_rate"] = 0.5

# ── Train new model ──────────────────────────────────────────────────
train3 = model_df_v3[model_df_v3["year"].isin([2021, 2022])]
test3  = model_df_v3[model_df_v3["year"] == 2023]

train3 = train3.dropna(subset=FEATURE_COLS_V3)
test3  = test3.dropna(subset=FEATURE_COLS_V3)

X_train3 = train3[FEATURE_COLS_V3]
y_train3 = train3[TARGET]
X_test3  = test3[FEATURE_COLS_V3]
y_test3  = test3[TARGET]

scaler3 = StandardScaler()
X_train3_scaled = scaler3.fit_transform(X_train3)
X_test3_scaled  = scaler3.transform(X_test3)

rf3 = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
rf3.fit(X_train3_scaled, y_train3)

# ── Backtest ─────────────────────────────────────────────────────────
test3_with_probs = test3.copy()
test3_with_probs["win_probability"] = rf3.predict_proba(X_test3_scaled)[:, 1]

print("\n=== V3 Model — 2023 Backtest ===\n")
total_correct = 0
for discipline in test3_with_probs["discipline"].unique():
    disc_df = test3_with_probs[test3_with_probs["discipline"] == discipline]
    disc_df = disc_df.sort_values("win_probability", ascending=False)
    top3_predicted = disc_df.head(3)["athlete_name"].tolist()
    top3_actual    = disc_df[disc_df["dl_top3"] == 1]["athlete_name"].tolist()
    hits = len(set(top3_predicted) & set(top3_actual))
    total_correct += hits
    print(f"── {discipline} ── {hits}/3")

print(f"\nTotal correct: {total_correct}")
print(f"Accuracy: {total_correct / (len(test3_with_probs['discipline'].unique()) * 3):.1%}")

# ── Feature importance ───────────────────────────────────────────────
print("\n=== Feature Importance ===")
importances = pd.Series(rf3.feature_importances_, index=FEATURE_COLS_V3)
print(importances.sort_values(ascending=False).to_string())

# ── Save new model ───────────────────────────────────────────────────
with open("outputs/model_rf.pkl", "wb") as f: pickle.dump(rf3, f)
with open("outputs/scaler.pkl",   "wb") as f: pickle.dump(scaler3, f)
with open("outputs/feature_cols.pkl", "wb") as f: pickle.dump(FEATURE_COLS_V3, f)
with open("outputs/model_accuracy.txt", "w") as f:
    f.write(str(round(total_correct / (len(test3_with_probs['discipline'].unique()) * 3) * 100, 1)))
print("\nNew model saved to outputs/")
accuracy_pct = round(total_correct / (len(test3_with_probs['discipline'].unique()) * 3) * 100, 1)
with open("outputs/model_accuracy.txt", "w") as f:
    f.write(str(accuracy_pct))
print(f"Accuracy saved: {accuracy_pct}%")

Working directory: C:\Users\rayen\athletics-predictor
Adding new features to training data...
Skipping h2h for training (not enough historical data)...

=== V3 Model — 2023 Backtest ===

── men_100m ── 1/3
── men_1500m ── 1/3
── men_200m ── 2/3
── men_400h ── 2/3
── men_800m ── 1/3
── men_LJ ── 0/3
── men_PV ── 2/3
── women_100m ── 2/3
── women_1500m ── 1/3
── women_200m ── 2/3
── women_400h ── 1/3
── women_800m ── 2/3
── women_PV ── 1/3

Total correct: 18
Accuracy: 46.2%

=== Feature Importance ===
season_rank             0.295485
season_percentile       0.266708
meets_count             0.112387
consistency             0.087586
weighted_season_best    0.049692
season_best             0.047224
career_best             0.045787
wind_adj_season_best    0.042348
age                     0.034105
yoy_improvement         0.013273
pb_gap                  0.005406

New model saved to outputs/
Accuracy saved: 46.2%
